# 06 — Gold: KPIs de `ecommerce_itens_pedido`

**Squad 2 — Negócio em Tempo Real | Dupla José Vieira & Tiago Ramos**

Lê a Silver de itens, calcula os KPIs da planilha e grava cada um em **Delta** (pasta `gold/` do container
da squad) e no **SQL Server** (schema `squad2`). KPIs são recalculados inteiros a cada execução
(**overwrite**); `gold_hist_kpis_itens_pedido` guarda uma foto por execução (**append**).

| Regra | Tabela | Alerta |
|---|---|---|
| N6 | `gold_kpi_skus_unicos_1h` | SKUs únicos na hora caem mais de `QUEDA_SKUS`% frente à média das 6 horas anteriores |
| N7 | `gold_kpi_top5_skus_30min` | o SKU líder concentra mais de `LIMITE_CONCENTRACAO`% das unidades da janela |
| N8 | `gold_kpi_receita_itens_pedido_1h` | desconto acima de 25% da receita bruta na hora |
| N9 | — | regra de bloqueio: aplicada na Silver (`flag_desconto`), contada em `gold_qualidade_itens_pedido` |
| N10 | `gold_kpi_itens_por_pedido_1h` | média abaixo de 3 itens por pedido na hora |

Para o dashboard, além dos KPIs:

| Tabela | Grão | Uso |
|---|---|---|
| `gold_fato_itens_pedido` | um item válido (Silver) | gráficos filtráveis por período, SKU, faixa de preço, desconto |
| `gold_qualidade_itens_pedido` | dia de ingestão × regra | página de qualidade: violações contra o total analisado |

"Item" nos KPIs é uma linha da tabela (um SKU dentro do pedido); unidades são a soma de `quantidade`.

In [ ]:
%run ./00_setup_config

## Parâmetros

Os limiares de N6 e N7 não foram definidos pelo negócio: valores provisórios.

In [ ]:
from datetime import timedelta
from pyspark.sql import functions as F, Window

QUEDA_SKUS, LIMITE_CONCENTRACAO = 50.0, 30.0           # provisorios
LIMITE_DESCONTO, MIN_ITENS_PEDIDO = 25.0, 3            # definidos na planilha
HORAS_JANELA = 24                                      # quanto da Silver entra nas tabelas por hora

## Leitura da Silver e instante de referência

Os KPIs são medidos a partir de agora. Fora de uma janela de tempo real a Silver não recebe itens novos e
tudo daria zero; nesse caso a referência passa a ser o último pedido da Silver, e o notebook avisa.

In [ ]:
silver = ler_delta(CAMINHO_SILVER)
if silver is None:
    raise ValueError(f"Silver vazia em {CAMINHO_SILVER}: rodar Bronze e Silver antes.")
itens = silver.withColumn("dt_local", F.expr(f"convert_timezone('{FUSO_NEGOCIO}', dt_pedido)"))

agora = spark.range(1).select(F.expr(f"convert_timezone('{FUSO_NEGOCIO}', current_timestamp())")).first()[0]
ultimo = itens.agg(F.max("dt_local")).first()[0]
ntz = lambda d: F.lit(d.strftime("%Y-%m-%d %H:%M:%S.%f")).cast("timestamp_ntz")  # texto: datetime sem fuso seria convertido pelo fuso da maquina
REF = agora if (agora - ultimo).total_seconds() <= HORAS_JANELA * 3600 else ultimo
print(f"Itens na Silver: {itens.count()} | ultimo pedido: {ultimo} | agora: {agora:%Y-%m-%d %H:%M:%S}")
print(f"Referencia dos KPIs: {REF}" + ("" if REF == agora else "  [aviso] sem itens nas ultimas horas: usando o ultimo pedido"))

ref = ntz(REF)
recentes = itens.filter((F.col("dt_local") > ntz(REF - timedelta(hours=HORAS_JANELA))) & (F.col("dt_local") <= ref))
ultima_hora = itens.filter((F.col("dt_local") > ntz(REF - timedelta(hours=1))) & (F.col("dt_local") <= ref))
por_hora = lambda df: df.groupBy(F.window("dt_local", "1 hour").alias("w"))
janela = [F.col("w.start").alias("inicio_janela"), F.col("w.end").alias("fim_janela")]

## N6 — SKUs únicos vendidos por hora

Uma linha por hora das últimas `HORAS_JANELA` horas. Queda brusca na diversidade indica problema no catálogo
ou no buscador; a comparação é com a média das 6 horas anteriores que tiveram venda.

In [ ]:
anteriores = Window.orderBy("inicio_janela").rowsBetween(-6, -1)
kpi_skus = (por_hora(recentes).agg(F.countDistinct("sku").alias("skus_unicos"), F.sum("quantidade").alias("unidades")).select(*janela, "skus_unicos", "unidades")
    .withColumn("media_6h_anteriores", F.round(F.avg("skus_unicos").over(anteriores), 1))
    .withColumn("alerta_queda", F.coalesce(F.col("skus_unicos") < F.col("media_6h_anteriores") * (1 - QUEDA_SKUS / 100), F.lit(False))))
skus_1h = ultima_hora.agg(F.countDistinct("sku")).first()[0]
print(f"SKUs unicos na ultima hora (ate {REF:%H:%M}): {skus_1h}")
display(kpi_skus.orderBy(F.desc("inicio_janela")))

## N7 — Top 5 SKUs na janela de 30 minutos

Ranking por unidades nos 30 minutos até a referência. Concentração anormal num SKU pode ser bot, scraper
ou promoção viral.

In [ ]:
ult30 = itens.filter((F.col("dt_local") > ntz(REF - timedelta(minutes=30))) & (F.col("dt_local") <= ref))
unidades_30 = ult30.agg(F.sum("quantidade")).first()[0] or 0
kpi_top5 = (ult30.groupBy("sku").agg(F.sum("quantidade").alias("unidades"), F.round(F.sum("receita_liquida"), 2).alias("receita_liquida"))
    .withColumn("posicao", F.row_number().over(Window.orderBy(F.desc("unidades"), "sku"))).filter("posicao <= 5")
    .withColumn("pct_unidades", F.round(F.col("unidades") / F.lit(max(unidades_30, 1)) * 100, 1))
    .withColumn("alerta_concentracao", (F.col("posicao") == 1) & (F.col("pct_unidades") > LIMITE_CONCENTRACAO))
    .select(ntz(REF - timedelta(minutes=30)).alias("inicio_janela"), ref.alias("fim_janela"), "posicao", "sku", "unidades", "pct_unidades", "receita_liquida", "alerta_concentracao"))
print(f"Unidades na janela de 30 min: {unidades_30}")
display(kpi_top5.orderBy("posicao"))

## N8 — Receita bruta x líquida por hora

Bruta = preço × quantidade; líquida = (preço − desconto) × quantidade. Alerta quando o desconto passa de
25% da bruta.

In [ ]:
kpi_receita = (por_hora(recentes).agg(F.round(F.sum("receita_bruta"), 2).alias("receita_bruta"), F.round(F.sum("receita_liquida"), 2).alias("receita_liquida"))
    .select(*janela, "receita_bruta", "receita_liquida")
    .withColumn("desconto", F.round(F.col("receita_bruta") - F.col("receita_liquida"), 2))
    .withColumn("pct_desconto", F.round(F.col("desconto") / F.col("receita_bruta") * 100, 1))
    .withColumn("alerta_desconto", F.coalesce(F.col("pct_desconto") > LIMITE_DESCONTO, F.lit(False))))
display(kpi_receita.orderBy(F.desc("inicio_janela")))

## N10 — Itens por pedido por hora

Queda abrupta pode indicar carrinho sendo esvaziado no checkout. Alerta abaixo de 3 itens por pedido.

In [ ]:
kpi_itens_pedido = (por_hora(recentes).agg(F.countDistinct("id_pedido").alias("pedidos"), F.count("*").alias("itens"), F.sum("quantidade").alias("unidades"))
    .select(*janela, "pedidos", "itens", "unidades")
    .withColumn("itens_por_pedido", F.round(F.col("itens") / F.col("pedidos"), 2))
    .withColumn("unidades_por_pedido", F.round(F.col("unidades") / F.col("pedidos"), 2))
    .withColumn("alerta_carrinho", F.col("itens_por_pedido") < MIN_ITENS_PEDIDO))
display(kpi_itens_pedido.orderBy(F.desc("inicio_janela")))

## Histórico (append)

Uma linha por KPI a cada execução, com o valor da **última hora** (ou dos últimos 30 min, no N7) até a
referência. É o que permite acompanhar a evolução entre execuções.

In [ ]:
h = ultima_hora.agg(F.countDistinct("sku").alias("skus"), F.sum("receita_bruta").alias("bruta"), F.sum("receita_liquida").alias("liquida"),
                    F.countDistinct("id_pedido").alias("pedidos"), F.count("*").alias("itens")).first()
pct_desc = round((1 - h["liquida"] / h["bruta"]) * 100, 1) if h["bruta"] else None
itens_ped = round(h["itens"] / h["pedidos"], 2) if h["pedidos"] else None
linhas_hist = [("skus_unicos_1h", "-", float(h["skus"]), False),
               ("receita_bruta_1h", "-", float(h["bruta"] or 0), False),
               ("receita_liquida_1h", "-", float(h["liquida"] or 0), False),
               ("pct_desconto_1h", "-", pct_desc, bool(pct_desc and pct_desc > LIMITE_DESCONTO)),
               ("itens_por_pedido_1h", "-", itens_ped, bool(itens_ped is not None and itens_ped < MIN_ITENS_PEDIDO))]
linhas_hist += [(f"top5_skus_30min_{r['posicao']}", r["sku"], float(r["unidades"]), r["alerta_concentracao"]) for r in kpi_top5.collect()]
hist = (spark.createDataFrame(linhas_hist, "kpi string, dimensao string, valor double, alerta boolean")
             .select(F.current_timestamp().alias("calculado_em"), ref.alias("referencia"), "*"))
display(hist)

## Fato de itens (para o dashboard)

Um item válido por linha, com as colunas que o dashboard usa para filtrar e agrupar. As métricas são
calculadas no próprio dashboard, em qualquer recorte.

In [ ]:
DIAS = F.element_at(F.array(*[F.lit(d) for d in ["dom", "seg", "ter", "qua", "qui", "sex", "sáb"]]), F.dayofweek("dt_local"))
fato = itens.select("id_item_pedido", "id_pedido", "sku", F.col("dt_local").alias("dt_pedido"), F.to_date("dt_local").alias("data"),
                    F.hour("dt_local").alias("hora"), DIAS.alias("dia_semana"), "quantidade", "preco_unitario", "desconto_aplicado",
                    "receita_bruta", "receita_liquida", (F.col("desconto_aplicado") > 0).alias("tem_desconto"),
                    F.when(F.col("preco_unitario") < 20, "1. até 20").when(F.col("preco_unitario") < 50, "2. 20–50")
                     .when(F.col("preco_unitario") < 150, "3. 50–150").otherwise("4. 150+").alias("faixa_preco"))
display(fato.orderBy(F.desc("dt_pedido")).limit(10))

## Qualidade dos dados (para o dashboard)

Por dia de ingestão: linhas analisadas, aprovadas, barradas e violações de cada regra, em formato longo
(uma linha por regra). Itens ainda na espera entram como `aguardando`.

In [ ]:
quar, esp = ler_delta(CAMINHO_QUARENTENA), ler_delta(CAMINHO_ESPERA)
REGRAS = {"flag_schema": "T1 schema", "flag_pedido_reprovado": "T2 pedido reprovado", "flag_sem_pedido": "T2 sem pedido", "flag_quantidade": "T3 quantidade", "flag_preco": "T4 preço",
          "flag_duplicado": "T5 duplicado", "flag_desconto": "N9 desconto > preço", "flag_desconto_negativo": "Extra desconto < 0"}
dia = F.to_date(F.expr(f"convert_timezone('{FUSO_NEGOCIO}', bronze_ingested_at)")).alias("data_ingestao")
contar = lambda df, nome: (df.groupBy(dia).agg(F.count("*").alias(nome)) if df is not None else spark.createDataFrame([], f"data_ingestao date, {nome} bigint"))
totais = (contar(silver, "aprovadas").join(contar(quar, "barradas"), "data_ingestao", "full").join(contar(esp, "aguardando"), "data_ingestao", "full")
          .fillna(0).withColumn("analisadas", F.col("aprovadas") + F.col("barradas") + F.col("aguardando")))
if quar is None:
    violacoes = spark.createDataFrame([], "data_ingestao date, regra string, violacoes bigint")
else:
    violacoes = (quar.select(dia, F.explode(F.array(*[F.struct(F.lit(r).alias("regra"), F.col(n).cast("int").alias("v")) for n, r in REGRAS.items()])).alias("x"))
                     .groupBy("data_ingestao", "x.regra").agg(F.sum("x.v").alias("violacoes")))
qualidade = (totais.join(violacoes, "data_ingestao", "left").fillna({"regra": "-", "violacoes": 0})
                   .withColumn("pct_violacoes", F.round(F.col("violacoes") / F.col("analisadas") * 100, 2)))
display(qualidade.orderBy(F.desc("data_ingestao"), "regra"))

## Gravação: Delta + SQL Server

Delta na pasta `gold/` do container da squad; SQL Server pelo conector nativo `sqlserver`. Uma falha numa
tabela não interrompe as outras; o resumo no fim lista o que falhou.

In [ ]:
saidas = {"gold_kpi_skus_unicos_1h": (kpi_skus, "overwrite"), "gold_kpi_top5_skus_30min": (kpi_top5, "overwrite"),
          "gold_kpi_receita_itens_pedido_1h": (kpi_receita, "overwrite"), "gold_kpi_itens_por_pedido_1h": (kpi_itens_pedido, "overwrite"),
          "gold_hist_kpis_itens_pedido": (hist, "append"), "gold_fato_itens_pedido": (fato, "overwrite"), "gold_qualidade_itens_pedido": (qualidade, "overwrite")}
falhas = []
for nome, (df, modo) in saidas.items():
    df = df.select([F.col(c).cast("timestamp").alias(c) if t == "timestamp_ntz" else F.col(c) for c, t in df.dtypes])  # SQL Server nao tem TIMESTAMP_NTZ
    destinos = [("delta", lambda: df.write.format("delta").options(**adls_options).mode(modo).option("overwriteSchema", str(modo == "overwrite").lower()).save(f"{BASE}/gold/{nome}")),
                ("sqlserver", lambda: df.write.format("sqlserver").options(**opcoes_sqlserver(f"{SQUAD}.{nome}")).mode(modo).save())]
    for destino, gravar in destinos:
        try:
            gravar()
        except Exception as e:
            falhas.append((nome, destino, str(e).splitlines()[0][:200]))
    print(f"{nome:36} {modo:9} {df.count():>6} linhas")
for f in falhas:
    print("[falha]", *f)
if falhas:
    raise RuntimeError(f"{len(falhas)} gravacao(oes) falharam")

## Conferência no SQL Server

In [ ]:
for nome in saidas:
    print(f"{SQUAD}.{nome:36} {spark.read.format('sqlserver').options(**opcoes_sqlserver(f'{SQUAD}.{nome}')).load().count():>6} linhas no SQL Server")
alertas = hist.filter("alerta").select("kpi", "dimensao", "valor")
print(f"\nAlertas ativos na referencia {REF}: {alertas.count()}"); display(alertas)

## Resumo

> **Base:** execução de 13/10/2026. Sem janela de tempo real aberta, a referência foi o último pedido da
> Silver, **08/10/2026 16:59**; as tabelas por hora cobrem 08/10 das 10h às 17h. Revisar após novas janelas.

**Nenhum alerta disparou.** Os itens válidos somam 9.196 linhas, 1.813 pedidos, 2.512 SKUs, R$ 494,7 mil de
receita bruta e R$ 485,2 mil de líquida.

- **N6 — diversidade:** de 208 a 482 SKUs distintos por hora; última hora com 257, abaixo da média das
  anteriores (344) mas longe da queda de 50% que alertaria. O pico foi às 14h, junto com o pico de pedidos.
- **N7 — concentração:** o líder dos últimos 30 minutos (desinfetante Lysoform) tem só 2,5% das unidades.
  Venda pulverizada, sem sinal de bot ou promoção viral.
- **N8 — desconto:** pesa de 1,1% a 1,5% da receita bruta na maior parte das horas; às 14h subiu para **4,6%**,
  o único ponto fora do padrão, ainda bem abaixo do limite de 25%. Vale olhar quais SKUs puxaram esse desconto.
- **N10 — cesta:** estável entre 4,2 e 4,6 itens (6,2 a 6,9 unidades) por pedido, acima do mínimo de 3.
  O checkout não mostra sinal de carrinho sendo esvaziado.

Leitura geral: em 08/10 a operação esteve estável; volume, diversidade e cesta sobem e descem juntos ao longo
do dia, o que é padrão de demanda, não de falha. O ponto de atenção não está nos KPIs, e sim na qualidade
(ver `05`): 31% dos itens ficam de fora destes números.